Week 13 · Day 2 — Causal Language Modeling (GPT-Style)
Why this matters

GPT models learn by predicting the next word given all previous words. This is the core of how chatbots and text generators work. Today you’ll build a tiny GPT-like training loop on a toy dataset.

Theory Essentials

Causal LM = predict the next token using only past context.

Requires a causal mask so the model cannot see future tokens.

Training loop: input sequence → predict next word → compute loss vs true next word.

Hugging Face gives ready-made datasets + models to speed up.

Output is probabilistic → sampling controls randomness.

In [19]:
# Setup
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer, TextDataset, DataCollatorForLanguageModeling, Trainer, TrainingArguments

# Load GPT-2 tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Tiny toy dataset (you can replace with your own .txt file)
text = """AI is transforming the world.
AI will shape the future.
AI is powerful but needs careful use.
AI and ethics will be necessary for proper world functioning.
AI without proper ethics can be devastating.
"""
with open("toy.txt", "w") as f:
    f.write(text)

dataset = TextDataset(
    tokenizer=tokenizer,
    file_path="toy.txt",
    block_size=32 #16
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # <-- Causal LM (not masked LM)
)

training_args = TrainingArguments(
    output_dir="./gpt_toy",
    overwrite_output_dir=True,
    per_device_train_batch_size=2,
    num_train_epochs=10, # 3
    logging_steps=5,
    save_steps=20,
    save_total_limit=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset,
)

trainer.train()

# Generate text from fine-tuned model
inputs = tokenizer("AI is", return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=10, pad_token_id=tokenizer.eos_token_id)
print("Generated:", tokenizer.decode(outputs[0]))


c:\AI-Mastery\venv\Lib\site-packages\transformers\data\datasets\language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(
c:\AI-Mastery\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
5,3.069400
10,1.770600


Generated: AI is a global leader in the field of health and wellness


1) Core (10–15 min)
Task: Train on the provided toy.txt and observe the generated text.

AI is a global leader in the field of healthcare.

2) Practice (10–15 min)
Task: Add new sentences about “AI and ethics” to toy.txt. Retrain. Compare outputs.

In [18]:
inputs = tokenizer("AI and ethics", return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=20, pad_token_id=tokenizer.eos_token_id)
print("Generated:", tokenizer.decode(outputs[0]))

Generated: AI and ethics.

The report also found that the government has been "trying to make the government look


3) Stretch (optional, 10–15 min)
Task: Change block_size to 32 and num_train_epochs to 5. Check impact on loss and generations.

With 10 epochs training loss: 1.77

Mini-Challenge (≤40 min)

Build a “toy GPT” text generator notebook.

Dataset: at least 10 custom sentences.

Train for ≥3 epochs.

Generate 3 completions starting with the same prompt.

Acceptance Criteria

Training runs without error.

Loss decreases across epochs.

Generated sentences reflect training text.

In [21]:



text = """The sun sets behind the mountains every evening.

My cat sleeps on the warm laptop when I study.

Tomorrow, we will run five kilometers in the park.

Artificial intelligence is changing the way humans work.

Please hand me the red notebook on the desk.

Coffee tastes better when shared with a friend.

She whispered a secret, but nobody heard.

Robots can learn tasks by observing people.

The river flows quietly under the old stone bridge.

I forgot my umbrella, so I got wet in the rain.
"""
with open("toy2.txt", "w") as f:
    f.write(text)

dataset = TextDataset(
    tokenizer=tokenizer,
    file_path="toy2.txt",
    block_size=32 #16
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # <-- Causal LM (not masked LM)
)

training_args = TrainingArguments(
    output_dir="./gpt_toy",
    overwrite_output_dir=True,
    per_device_train_batch_size=2,
    num_train_epochs=10, # 3
    logging_steps=5,
    save_steps=20,
    save_total_limit=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset,
)

trainer.train()

# Generate text from fine-tuned model

for i in range(3):
    inputs = tokenizer("My life is ", return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=20, pad_token_id=tokenizer.eos_token_id)
    print("Generated:", tokenizer.decode(outputs[0]))


c:\AI-Mastery\venv\Lib\site-packages\transformers\data\datasets\language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(
c:\AI-Mastery\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
5,0.622200
10,0.330200
15,0.304500
20,0.250100


Generated: My life is  a journey.

I will learn the art of the sword by observing it.


Generated: My life is icky.

Please hand me the red notebook.

Please write me a note.

Generated: My life is  truly changing.
I will soon be able to work with my best friend.
I


Notes / Key Takeaways

GPT = causal language modeling (predict next token).

Future tokens are hidden with a causal mask.

Small datasets → model quickly memorizes.

Output depends on sampling (temperature, top-k).

Hugging Face Trainer makes experiments simple.

Reflection

Why does causal LM forbid access to future tokens?

How does dataset size affect generation quality?

Because during generation the model only knows what has been written so far. If it could “peek” at future tokens during training, it would cheat and not learn to predict correctly. Masking future tokens makes training match real usage: left-to-right prediction.

Larger datasets expose the model to more vocabulary, grammar, and world knowledge. With small data, the model memorizes and repeats (overfits); with large diverse data, it learns patterns that produce fluent, coherent, and creative continuations.